In [6]:
import time
import os
import psutil
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, average_precision_score
from pycaret.classification import setup, compare_models, predict_model, pull
import h2o
from h2o.automl import H2OAutoML
from flaml import AutoML

In [7]:
%pip install "pandas==2.1.4" "numpy==1.26.4" "scipy==1.11.4" "joblib==1.3.2" "scikit-learn==1.4.2" "pycaret==3.3.2"


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [8]:
df = pd.read_csv("../data/creditcard.csv")

In [9]:
# =========================
# COLUNAS
# =========================

X = df.drop(['Class'], axis=1)
y = df['Class'].astype(int)


In [10]:
# =========================
# FUNÇÃO AUXILIAR DE RECURSOS
# =========================
def get_memory_usage_mb():
    """Retorna o uso atual de memória RAM do processo em Megabytes."""
    process = psutil.Process(os.getpid())
    return round(process.memory_info().rss / (1024 * 1024), 2)

# =========================
# INSTALAR PYCARET
# =========================
# %pip install pycaret psutil

for SEED in [42, 101, 223, 302, 404]:
    print(f"\n================================")
    print(f"EXECUTANDO COM SEED: {SEED}")
    print(f"================================")

    # =========================
    # DIVISÃO TREINO/TESTE
    # =========================
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=SEED, stratify=y
    )
    
    # =========================
    # DATAFRAME FINAL
    # =========================
    train_df = pd.DataFrame(X_train)
    train_df['Class'] = y_train.values

    test_df = pd.DataFrame(X_test)
    test_df['Class'] = y_test.values

    # =========================
    # PYCARET + MÉTRICA COMPUTACIONAL (TREINO)
    # =========================
    train_df = train_df.reset_index(drop=True)
    test_df = test_df.reset_index(drop=True)

    print("Iniciando treinamento do PyCaret (setup + compare_models)...")
    mem_antes_treino = get_memory_usage_mb()
    inicio_treino = time.perf_counter()

    exp = setup(
        data=train_df,
        test_data=test_df,
        target="Class",
        session_id=SEED,
        verbose=False,
        index=False,
        fix_imbalance=False,
    )

    # Retorna os 10 melhores modelos treinados
    modelos_treinados = compare_models(n_select=10)

    fim_treino = time.perf_counter()
    tempo_treino_total = round(fim_treino - inicio_treino, 2)
    mem_depois_treino = get_memory_usage_mb()

    # =========================
    # LEADERBOARD INTERNO DO PYCARET
    # =========================
    print("\n===== LEADERBOARD (Validação PyCaret) =====")
    print(pull())
    leaderboard = pull()

    # =========================
    # MÉTRICAS DE TODOS OS MODELOS (No Test Set)
    # =========================
    resultados = []

    for model in modelos_treinados:
        model_name = model.__class__.__name__

        # Tempo de inferência no conjunto de teste (previsão + score)
        inicio_inferencia = time.perf_counter()
        preds_df = predict_model(model, data=test_df, verbose=False)
        tempo_inferencia = time.perf_counter() - inicio_inferencia

        y_true = test_df['Class'].values
        y_pred = preds_df['prediction_label'].astype(int).values

        # Métricas
        acc = round(accuracy_score(y_true, y_pred), 4)
        precision = round(precision_score(y_true, y_pred, pos_label=1, zero_division=0), 4)
        recall = round(recall_score(y_true, y_pred, pos_label=1, zero_division=0), 4)
        f1_weighted = round(f1_score(y_true, y_pred, pos_label=1, zero_division=0, average='weighted'), 4)
        f1 = round(f1_score(y_true, y_pred, pos_label=1, zero_division=0), 4)

        score_col = [col for col in preds_df.columns if 'score' in col]
        if score_col:
            pos_score_col = [c for c in score_col if '1' in str(c)]
            if pos_score_col:
                y_score = preds_df[pos_score_col[0]].values
            else:
                y_score = preds_df[score_col[0]].values
        else:
            y_score = y_pred

        average_precision = round(average_precision_score(y_true, y_score), 4)

        resultados.append({
            "Modelo": model_name,
            "Tipo": model_name,
            "Seed": SEED,
            "Accuracy": acc,
            "Precision": precision,
            "Recall": recall,
            "F1-score Weighted": f1_weighted,
            "F1-score": f1,
            "average_precision": average_precision,
            "Tempo Inferência (s)": round(tempo_inferencia, 4),
            "Tempo Treino Total (s)": tempo_treino_total,
            "Uso RAM Treino (MB)": mem_depois_treino
        })

    # =========================
    # RESULTADOS
    # =========================
    df_resultados = pd.DataFrame(resultados)

    print("\n===== MÉTRICAS DE TODOS OS MODELOS (SEED {}) =====".format(SEED))
    print(df_resultados.sort_values(by="average_precision", ascending=False).to_string(index=False))


EXECUTANDO COM SEED: 42
Iniciando treinamento do PyCaret (setup + compare_models)...


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
xgboost,Extreme Gradient Boosting,0.9996,0.9831,0.7915,0.9506,0.8617,0.8615,0.8662,0.3690
rf,Random Forest Classifier,0.9995,0.9501,0.7738,0.9470,0.8493,0.8491,0.8546,124.1070
et,Extra Trees Classifier,0.9995,0.9525,0.7687,0.9578,0.8508,0.8505,0.8568,1.6710
lda,Linear Discriminant Analysis,0.9994,0.9808,0.7587,0.8788,0.8129,0.8126,0.8155,0.0680
lr,Logistic Regression,0.9992,0.9633,0.6396,0.8646,0.7340,0.7336,0.7426,1.6810
ada,Ada Boost Classifier,0.9992,0.9724,0.7074,0.7947,0.7460,0.7456,0.7481,5.3280
dt,Decision Tree Classifier,0.9991,0.8867,0.7738,0.7477,0.7575,0.7571,0.7587,1.6840
ridge,Ridge Classifier,0.9989,0.9808,0.4138,0.8591,0.5574,0.5569,0.5951,0.0360
gbc,Gradient Boosting Classifier,0.9988,0.6010,0.4328,0.7367,0.5153,0.5148,0.5463,27.3190
knn,K Neighbors Classifier,0.9983,0.5887,0.0482,0.8750,0.0902,0.0900,0.1985,4.0700



===== LEADERBOARD (Validação PyCaret) =====
                                    Model  Accuracy     AUC  Recall   Prec.  \
xgboost         Extreme Gradient Boosting    0.9996  0.9831  0.7915  0.9506   
rf               Random Forest Classifier    0.9995  0.9501  0.7738  0.9470   
et                 Extra Trees Classifier    0.9995  0.9525  0.7687  0.9578   
lda          Linear Discriminant Analysis    0.9994  0.9808  0.7587  0.8788   
lr                    Logistic Regression    0.9992  0.9633  0.6396  0.8646   
ada                  Ada Boost Classifier    0.9992  0.9724  0.7074  0.7947   
dt               Decision Tree Classifier    0.9991  0.8867  0.7738  0.7477   
ridge                    Ridge Classifier    0.9989  0.9808  0.4138  0.8591   
gbc          Gradient Boosting Classifier    0.9988  0.6010  0.4328  0.7367   
knn                K Neighbors Classifier    0.9983  0.5887  0.0482  0.8750   
dummy                    Dummy Classifier    0.9983  0.5000  0.0000  0.0000   
svm    

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
rf,Random Forest Classifier,0.9995,0.9474,0.7661,0.9341,0.8385,0.8382,0.8440,21.5200
et,Extra Trees Classifier,0.9995,0.9485,0.7712,0.9430,0.8455,0.8452,0.8510,1.9550
xgboost,Extreme Gradient Boosting,0.9995,0.9737,0.7765,0.9462,0.8511,0.8509,0.8560,0.4350
lda,Linear Discriminant Analysis,0.9994,0.9791,0.7508,0.8572,0.7984,0.7981,0.8009,0.0700
ada,Ada Boost Classifier,0.9992,0.9673,0.6851,0.8149,0.7413,0.7409,0.7452,5.9480
lr,Logistic Regression,0.9991,0.9577,0.6067,0.8540,0.7063,0.7058,0.7178,1.4380
dt,Decision Tree Classifier,0.9991,0.8677,0.7358,0.7671,0.7489,0.7485,0.7498,1.7250
ridge,Ridge Classifier,0.9988,0.9791,0.4107,0.8248,0.5402,0.5397,0.5762,0.0350
gbc,Gradient Boosting Classifier,0.9987,0.5452,0.3781,0.7703,0.4828,0.4823,0.5229,30.2190
knn,K Neighbors Classifier,0.9983,0.6024,0.0529,0.7250,0.0979,0.0977,0.1918,8.8200



===== LEADERBOARD (Validação PyCaret) =====
                                    Model  Accuracy     AUC  Recall   Prec.  \
rf               Random Forest Classifier    0.9995  0.9474  0.7661  0.9341   
et                 Extra Trees Classifier    0.9995  0.9485  0.7712  0.9430   
xgboost         Extreme Gradient Boosting    0.9995  0.9737  0.7765  0.9462   
lda          Linear Discriminant Analysis    0.9994  0.9791  0.7508  0.8572   
ada                  Ada Boost Classifier    0.9992  0.9673  0.6851  0.8149   
lr                    Logistic Regression    0.9991  0.9577  0.6067  0.8540   
dt               Decision Tree Classifier    0.9991  0.8677  0.7358  0.7671   
ridge                    Ridge Classifier    0.9988  0.9791  0.4107  0.8248   
gbc          Gradient Boosting Classifier    0.9987  0.5452  0.3781  0.7703   
knn                K Neighbors Classifier    0.9983  0.6024  0.0529  0.7250   
dummy                    Dummy Classifier    0.9983  0.5000  0.0000  0.0000   
svm    

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
rf,Random Forest Classifier,0.9996,0.9478,0.7894,0.9627,0.8667,0.8665,0.8712,45.3270
et,Extra Trees Classifier,0.9996,0.9501,0.7869,0.9635,0.8656,0.8654,0.8702,1.6670
xgboost,Extreme Gradient Boosting,0.9996,0.9755,0.7945,0.9545,0.8668,0.8666,0.8704,0.3770
lda,Linear Discriminant Analysis,0.9994,0.9817,0.7590,0.8688,0.8082,0.8079,0.8107,0.0660
lr,Logistic Regression,0.9992,0.9600,0.6446,0.8639,0.7363,0.7359,0.7448,2.3790
dt,Decision Tree Classifier,0.9992,0.8831,0.7665,0.7750,0.7698,0.7694,0.7699,2.2060
ada,Ada Boost Classifier,0.9992,0.9713,0.7108,0.8310,0.7625,0.7621,0.7663,5.0680
gbc,Gradient Boosting Classifier,0.9991,0.7130,0.5647,0.8481,0.6688,0.6684,0.6865,34.9820
ridge,Ridge Classifier,0.9989,0.9817,0.4337,0.8555,0.5709,0.5705,0.6059,0.0330
knn,K Neighbors Classifier,0.9984,0.6053,0.0560,0.8500,0.1019,0.1018,0.2057,6.5970



===== LEADERBOARD (Validação PyCaret) =====
                                    Model  Accuracy     AUC  Recall   Prec.  \
rf               Random Forest Classifier    0.9996  0.9478  0.7894  0.9627   
et                 Extra Trees Classifier    0.9996  0.9501  0.7869  0.9635   
xgboost         Extreme Gradient Boosting    0.9996  0.9755  0.7945  0.9545   
lda          Linear Discriminant Analysis    0.9994  0.9817  0.7590  0.8688   
lr                    Logistic Regression    0.9992  0.9600  0.6446  0.8639   
dt               Decision Tree Classifier    0.9992  0.8831  0.7665  0.7750   
ada                  Ada Boost Classifier    0.9992  0.9713  0.7108  0.8310   
gbc          Gradient Boosting Classifier    0.9991  0.7130  0.5647  0.8481   
ridge                    Ridge Classifier    0.9989  0.9817  0.4337  0.8555   
knn                K Neighbors Classifier    0.9984  0.6053  0.0560  0.8500   
dummy                    Dummy Classifier    0.9983  0.5000  0.0000  0.0000   
svm    

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
rf,Random Forest Classifier,0.9995,0.9437,0.7715,0.9523,0.8506,0.8503,0.8560,22.9620
et,Extra Trees Classifier,0.9995,0.9462,0.7688,0.9523,0.8486,0.8483,0.8543,1.3550
xgboost,Extreme Gradient Boosting,0.9995,0.9780,0.7792,0.9489,0.8546,0.8544,0.8591,0.3140
lda,Linear Discriminant Analysis,0.9994,0.9796,0.7537,0.8685,0.8058,0.8054,0.8081,0.0740
lr,Logistic Regression,0.9992,0.9555,0.6367,0.8584,0.7287,0.7283,0.7376,1.5980
dt,Decision Tree Classifier,0.9992,0.8778,0.7559,0.7967,0.7725,0.7721,0.7740,2.7010
ada,Ada Boost Classifier,0.9991,0.9694,0.6420,0.8208,0.7178,0.7174,0.7241,5.8070
gbc,Gradient Boosting Classifier,0.9989,0.6470,0.4579,0.8263,0.5644,0.5640,0.5982,143.6700
ridge,Ridge Classifier,0.9988,0.9796,0.4035,0.8503,0.5420,0.5415,0.5821,0.0410
knn,K Neighbors Classifier,0.9983,0.5987,0.0381,0.6000,0.0707,0.0706,0.1457,5.7460



===== LEADERBOARD (Validação PyCaret) =====
                                    Model  Accuracy     AUC  Recall   Prec.  \
rf               Random Forest Classifier    0.9995  0.9437  0.7715  0.9523   
et                 Extra Trees Classifier    0.9995  0.9462  0.7688  0.9523   
xgboost         Extreme Gradient Boosting    0.9995  0.9780  0.7792  0.9489   
lda          Linear Discriminant Analysis    0.9994  0.9796  0.7537  0.8685   
lr                    Logistic Regression    0.9992  0.9555  0.6367  0.8584   
dt               Decision Tree Classifier    0.9992  0.8778  0.7559  0.7967   
ada                  Ada Boost Classifier    0.9991  0.9694  0.6420  0.8208   
gbc          Gradient Boosting Classifier    0.9989  0.6470  0.4579  0.8263   
ridge                    Ridge Classifier    0.9988  0.9796  0.4035  0.8503   
knn                K Neighbors Classifier    0.9983  0.5987  0.0381  0.6000   
dummy                    Dummy Classifier    0.9983  0.5000  0.0000  0.0000   
svm    

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
rf,Random Forest Classifier,0.9996,0.9427,0.7817,0.9541,0.8582,0.8580,0.8629,19.6760
et,Extra Trees Classifier,0.9996,0.9477,0.7893,0.9603,0.8656,0.8654,0.8700,5.1600
xgboost,Extreme Gradient Boosting,0.9996,0.9803,0.8021,0.9563,0.8705,0.8703,0.8747,28.0100
lda,Linear Discriminant Analysis,0.9994,0.9824,0.7742,0.8677,0.8176,0.8173,0.8190,0.1130
lr,Logistic Regression,0.9992,0.9580,0.6425,0.8829,0.7401,0.7398,0.7509,1.5460
dt,Decision Tree Classifier,0.9992,0.8806,0.7617,0.7559,0.7571,0.7567,0.7575,2.0480
ada,Ada Boost Classifier,0.9992,0.9717,0.7005,0.8157,0.7523,0.7519,0.7548,218.5180
ridge,Ridge Classifier,0.9989,0.9824,0.4167,0.8477,0.5537,0.5532,0.5909,0.0390
gbc,Gradient Boosting Classifier,0.9989,0.6167,0.4769,0.8247,0.5887,0.5882,0.6173,2878.2070
knn,K Neighbors Classifier,0.9984,0.6105,0.0585,0.8524,0.1070,0.1068,0.2127,5.1040



===== LEADERBOARD (Validação PyCaret) =====
                                    Model  Accuracy     AUC  Recall   Prec.  \
rf               Random Forest Classifier    0.9996  0.9427  0.7817  0.9541   
et                 Extra Trees Classifier    0.9996  0.9477  0.7893  0.9603   
xgboost         Extreme Gradient Boosting    0.9996  0.9803  0.8021  0.9563   
lda          Linear Discriminant Analysis    0.9994  0.9824  0.7742  0.8677   
lr                    Logistic Regression    0.9992  0.9580  0.6425  0.8829   
dt               Decision Tree Classifier    0.9992  0.8806  0.7617  0.7559   
ada                  Ada Boost Classifier    0.9992  0.9717  0.7005  0.8157   
ridge                    Ridge Classifier    0.9989  0.9824  0.4167  0.8477   
gbc          Gradient Boosting Classifier    0.9989  0.6167  0.4769  0.8247   
knn                K Neighbors Classifier    0.9984  0.6105  0.0585  0.8524   
dummy                    Dummy Classifier    0.9983  0.5000  0.0000  0.0000   
svm    

In [ ]:
# =========================
# FUNÇÃO AUXILIAR DE RECURSOS
# =========================
def get_memory_usage_mb():
    """Retorna o uso atual de memória RAM do processo em Megabytes."""
    process = psutil.Process(os.getpid())
    return round(process.memory_info().rss / (1024 * 1024), 2)

# =========================
# INSTALAR H2O
# =========================
# %pip install h2o psutil

# =========================
# INICIAR H2O
# =========================
h2o.init()

target_col = 'Class'

for SEED in [42, 101, 223, 302, 404]:
    print(f"\n================================")
    print(f"EXECUTANDO COM SEED: {SEED}")
    print(f"================================")

    # =========================
    # DIVISÃO TREINO/TESTE
    # =========================
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=SEED, stratify=y
    )

    # =========================
    # DATAFRAME FINAL
    # =========================
    train_df = pd.DataFrame(X_train)
    train_df['Class'] = y_train.values

    test_df = pd.DataFrame(X_test)
    test_df['Class'] = y_test.values

    # =========================
    # PARA H2O
    # =========================
    train_h2o = h2o.H2OFrame(train_df)
    test_h2o = h2o.H2OFrame(test_df)

    train_h2o['Class'] = train_h2o['Class'].asfactor()
    test_h2o['Class'] = test_h2o['Class'].asfactor()

    x = train_h2o.columns[:-1]

    # =========================
    # AUTO ML + MÉTRICA COMPUTACIONAL (TREINO)
    # =========================
    aml = H2OAutoML(
        seed=SEED,
        balance_classes=False,
    )

    print("Iniciando treinamento do H2O AutoML...")
    mem_antes_treino = get_memory_usage_mb()
    inicio_treino = time.perf_counter()

    aml.train(x=x, y=target_col, training_frame=train_h2o)

    fim_treino = time.perf_counter()
    tempo_treino_total = round(fim_treino - inicio_treino, 2)
    mem_depois_treino = get_memory_usage_mb()

    # =========================
    # LEADERBOARD
    # =========================
    print("\n===== LEADERBOARD =====")
    print(aml.leaderboard)

    # =========================
    # MÉTRICAS DE TODOS OS MODELOS
    # =========================
    leaderboard_df = aml.leaderboard.as_data_frame()
    print("Colunas disponíveis no leaderboard:", leaderboard_df.columns.tolist())
    
    # Identifica dinamicamente a coluna de tempo de treino disponível no leaderboard
    time_col = None
    for col in ["training_time_ms", "run_time", "runtime"]:
        if col in leaderboard_df.columns:
            time_col = col
            break
            
    if time_col:
        training_times = leaderboard_df.set_index("model_id")[time_col]
    else:
        training_times = pd.Series(0, index=leaderboard_df["model_id"])

    resultados = []

    for model_id in leaderboard_df['model_id']:
        model = h2o.get_model(model_id)

        # Tempo de inferência para cada modelo
        inicio_inferencia = time.perf_counter()
        preds = model.predict(test_h2o).as_data_frame()
        tempo_inferencia = time.perf_counter() - inicio_inferencia
        
        y_true = test_df[target_col].values
        y_pred = preds['predict'].astype(int).values
        y_score = preds['p1'].values

        # Métricas
        acc = round(accuracy_score(y_true, y_pred), 4)
        precision = round(precision_score(y_true, y_pred, pos_label=1, zero_division=0), 4)
        recall = round(recall_score(y_true, y_pred, pos_label=1, zero_division=0), 4)
        f1_weighted = round(f1_score(y_true, y_pred, pos_label=1, zero_division=0, average='weighted'), 4)
        f1 = round(f1_score(y_true, y_pred, pos_label=1, zero_division=0), 4)
        average_precision = round(average_precision_score(y_true, y_score), 4)
        
        tempo_treino_modelo_raw = training_times.get(model_id, 0)
        
        if time_col == "training_time_ms":
            tempo_treino_modelo_s = round(tempo_treino_modelo_raw / 1000, 4) if pd.notna(tempo_treino_modelo_raw) else 0.0
        else:
            tempo_treino_modelo_s = round(tempo_treino_modelo_raw, 4) if pd.notna(tempo_treino_modelo_raw) else 0.0

        resultados.append({
            "Modelo": model_id,
            "Tipo": model.algo,
            "Seed": SEED,
            "Accuracy": acc,
            "Precision": precision,
            "Recall": recall,
            "F1-score": f1,
            "F1-score Weighted": f1_weighted,
            "average_precision": average_precision,
            "Tempo Inferência (s)": round(tempo_inferencia, 4),
            "Tempo Treino Modelo (s)": tempo_treino_modelo_s,
            "Tempo Treino Total AutoML (s)": tempo_treino_total,
            "Uso RAM Treino (MB)": mem_depois_treino
        })

    # =========================
    # RESULTADO FINAL
    # =========================
    df_resultados = pd.DataFrame(resultados)

    print("\n===== MÉTRICAS DE TODOS OS MODELOS SEED {} =====".format(SEED))
    print(df_resultados.sort_values(by="average_precision", ascending=False).to_string(index=False))

    h2o.remove_all()
    h2o.init()

Checking whether there is an H2O instance running at http://localhost:54321..... not found.
Attempting to start a local H2O server...
  Java Version: openjdk version "17.0.20" 2026-07-21; OpenJDK Runtime Environment Homebrew (build 17.0.20+0); OpenJDK 64-Bit Server VM Homebrew (build 17.0.20+0, mixed mode, sharing)
  Starting server from /opt/homebrew/lib/python3.11/site-packages/h2o/backend/bin/h2o.jar
  Ice root: /var/folders/x_/pw0p233x3n5_ny9bsbpb0ldc0000gn/T/tmpnwgmbwr8
  JVM stdout: /var/folders/x_/pw0p233x3n5_ny9bsbpb0ldc0000gn/T/tmpnwgmbwr8/h2o_lannaces_started_from_python.out
  JVM stderr: /var/folders/x_/pw0p233x3n5_ny9bsbpb0ldc0000gn/T/tmpnwgmbwr8/h2o_lannaces_started_from_python.err
  Server is running at http://127.0.0.1:54321
Connecting to H2O server at http://127.0.0.1:54321 ... successful.


H2O_cluster_uptime:,02 secs
H2O_cluster_timezone:,America/Sao_Paulo
H2O_data_parsing_timezone:,UTC
H2O_cluster_version:,3.46.0.12
H2O_cluster_version_age:,2 months and 30 days
H2O_cluster_name:,H2O_from_python_lannaces_iq0r5a
H2O_cluster_total_nodes:,1
H2O_cluster_free_memory:,4 Gb
H2O_cluster_total_cores:,10
H2O_cluster_allowed_cores:,10
H2O_cluster_status:,"locked, healthy"



+------------------------------------------------------------------+
| You are running the community edition of H2O-3 OSS.              |
|                                                                  |
| For commercial use, H2O-3 Secure is now recommended.             |
| This includes production support, CVE fixes, multi-node scaling, |
| model artifact extraction, and more.                             |
| See h2o.ai/h2o-3/oss-vs-secure for additional details.           |
| Contact enterprise@h2o.ai to upgrade.                            |
+------------------------------------------------------------------+

EXECUTANDO COM SEED: 42
Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
Iniciando treinamento do H2O AutoML...
AutoML progress: |
12:20:04.770: AutoML: XGBoost is not available; skipping it.

███████████████████████████████████████████

KeyError: 'run_time'

In [20]:
# =========================
# FUNÇÃO AUXILIAR DE RECURSOS
# =========================
def get_memory_usage_mb():
    """Retorna o uso atual de memória RAM do processo em Megabytes."""
    process = psutil.Process(os.getpid())
    return round(process.memory_info().rss / (1024 * 1024), 2)

# =========================
# INSTALAR FLAML 
# =========================
# %pip install "flaml[automl]" psutil

for SEED in [42, 101, 223, 302, 404]:
    print(f"\n================================")
    print(f"EXECUTANDO COM SEED: {SEED}")
    print(f"================================")

    # =========================
    # DIVISÃO TREINO/TESTE
    # =========================
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=SEED, stratify=y
    )

    # =========================
    # TREINAMENTO COM FLAML + MÉTRICA COMPUTACIONAL
    # =========================
    automl = AutoML()
    settings = {
        "task": "classification",
        "log_file_name": "flaml.log",
        "seed": SEED
    }

    print("Iniciando treinamento com FLAML...")
    mem_antes_treino = get_memory_usage_mb()
    inicio_treino = time.perf_counter()

    automl.fit(X_train=X_train, y_train=y_train, **settings)

    fim_treino = time.perf_counter()
    tempo_treino_total = round(fim_treino - inicio_treino, 2)
    mem_depois_treino = get_memory_usage_mb()

    # =========================
    # RESULTADOS DO MELHOR MODELO
    # =========================
    print("\n===== MELHOR RESULTADO FLAML =====")
    print("Melhor modelo:", automl.best_estimator)
    print("Melhores parâmetros:", automl.best_config)
    print("Melhor loss:", automl.best_loss)

    # =========================
    # SELECIONAR OS 5 MELHORES ESTIMADORES
    # =========================
    perdas_por_estimador = automl.best_loss_per_estimator
    melhores_estimadores = sorted(
        perdas_por_estimador.items(),
        key=lambda item: item[1]
    )[:5]

    if not melhores_estimadores:
        raise RuntimeError("O FLAML não encontrou estimadores avaliados.")

    print("\n===== 5 MELHORES ESTIMADORES (VALIDAÇÃO FLAML) =====")
    print(pd.DataFrame(
        melhores_estimadores,
        columns=["Modelo", "Loss validação"]
    ).to_string(index=False))

    # =========================
    # MÉTRICAS DOS 5 MELHORES NO TESTE (COM RECONSTRUÇÃO SEGURA)
    # =========================
    resultados = []
    y_true = y_test.to_numpy()

    for rank, (nome_modelo, loss_validacao) in enumerate(melhores_estimadores, start=1):
        try:
            # Tenta pegar o modelo nativo salvo pelo FLAML
            modelo = automl.best_model_for_estimator(nome_modelo)
            
            # Se o modelo veio None ou sem estrutura interna treinada, recriamos via config otimizada do FLAML
            if modelo is None or getattr(modelo, "_model", None) is None:
                config_params = automl.best_config_per_estimator.get(nome_modelo, {})
                
                # Mapeamento limpo para instanciar e treinar o estimador com os melhores hiperparâmetros descobertos
                if nome_modelo in ['xgboost', 'xgb_limitdepth']:
                    from xgboost import XGBClassifier
                    modelo = XGBClassifier(**config_params, random_state=SEED, n_jobs=-1)
                elif nome_modelo == 'lgbm':
                    from lightgbm import LGBMClassifier
                    modelo = LGBMClassifier(**config_params, random_state=SEED, n_jobs=-1, verbose=-1)
                elif nome_modelo == 'rf':
                    from sklearn.ensemble import RandomForestClassifier
                    modelo = RandomForestClassifier(**config_params, random_state=SEED, n_jobs=-1)
                elif nome_modelo == 'extra_tree':
                    from sklearn.ensemble import ExtraTreesClassifier
                    modelo = ExtraTreesClassifier(**config_params, random_state=SEED, n_jobs=-1)
                elif nome_modelo == 'sgd':
                    from sklearn.linear_model import SGDClassifier
                    modelo = SGDClassifier(**config_params, random_state=SEED, loss='log_loss')
                elif nome_modelo == 'lrl1':
                    from sklearn.linear_model import LogisticRegression
                    modelo = LogisticRegression(**config_params, random_state=SEED, penalty='l1', solver='liblinear')
                else:
                    print(f"Aviso: Estimador {nome_modelo} não mapeado para reconstrução manual.")
                    continue
                
                # Treina o modelo reconstruído com os dados de treino
                modelo.fit(X_train, y_train)
            else:
                # Extrai o estimador interno scikit-learn/XGBoost se estiver envelopado pelo FLAML
                if hasattr(modelo, "estimator"):
                    modelo = modelo.estimator
                elif hasattr(modelo, "_model"):
                    modelo = modelo._model

            # Tempo de inferência para cada modelo
            inicio_inferencia = time.perf_counter()
            y_pred = modelo.predict(X_test)

            if hasattr(modelo, "predict_proba"):
                probabilidades = modelo.predict_proba(X_test)
                classes = list(modelo.classes_)
                y_score = probabilidades[:, classes.index(1)]
            elif hasattr(modelo, "decision_function"):
                y_score = modelo.decision_function(X_test)
                if list(modelo.classes_).index(1) == 0:
                    y_score = -y_score
            else:
                y_score = y_pred
                
            tempo_inferencia = time.perf_counter() - inicio_inferencia

            resultados.append({
                "Posição": rank,
                "Modelo": nome_modelo,
                "Loss validação": round(float(loss_validacao), 4),
                "Accuracy": round(accuracy_score(y_true, y_pred), 4),
                "Precision": round(precision_score(y_true, y_pred, pos_label=1, zero_division=0), 4),
                "Recall": round(recall_score(y_true, y_pred, pos_label=1, zero_division=0), 4),
                "F1-score Weighted": round(f1_score(y_true, y_pred, average="weighted", zero_division=0), 4),
                "F1-score": round(f1_score(y_true, y_pred, pos_label=1, zero_division=0), 4),
                "Average Precision": round(average_precision_score(y_true, y_score), 4),
                "Seed": SEED,
                "Tempo Inferência (s)": round(tempo_inferencia, 4),
                "Tempo Treino Total (s)": tempo_treino_total,
                "Uso RAM Treino (MB)": mem_depois_treino
            })
        except Exception as e:
            print(f"Erro ao processar o modelo '{nome_modelo}': {e}")
            continue

    df_resultados_flaml = pd.DataFrame(resultados)
    print("\n===== MÉTRICAS DOS 5 MELHORES MODELOS FLAML (TESTE) =====")
    print(df_resultados_flaml.to_string(index=False))


EXECUTANDO COM SEED: 42
Iniciando treinamento com FLAML...
[flaml.automl.logger: 11-09 12:55:17] {2470} INFO - task = classification
[flaml.automl.logger: 11-09 12:55:17] {2481} INFO - Evaluation method: cv
[flaml.automl.logger: 11-09 12:55:17] {2584} INFO - Minimizing error metric: 1-roc_auc
[flaml.automl.logger: 11-09 12:55:17] {2607} WARNING - No search budget is provided via time_budget or max_iter. Training only one model per estimator. Zero-shot AutoML is used for certain tasks and estimators. To tune hyperparameters for each estimator, please provide budget either via time_budget or max_iter.
[flaml.automl.logger: 11-09 12:55:17] {2701} INFO - List of ML learners in AutoML Run: ['xgboost', 'lgbm', 'rf', 'extra_tree', 'xgb_limitdepth', 'sgd', 'lrl1']
[flaml.automl.logger: 11-09 12:55:17] {3006} INFO - iteration 0, current learner xgboost
[flaml.automl.logger: 11-09 16:54:43] {3141} INFO - Estimated sufficient time budget=10000s. Estimated necessary time budget=10s.
[flaml.automl